In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1025.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1047.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit486.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit728.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1013.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit282.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit687.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit889.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit554.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit772.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit305.txt
/kaggle/input/yol

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall

In [2]:
from ultralytics import YOLO
import time, pandas as pd
from tabulate import tabulate
import os
# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed across ablations)
base_params = {
    "batch": 32,
    "lr0": 0.01,
    "optimizer": "SGD",
    "imgsz": 640,
    "weight_decay": 0.0005
}

# results log file
results_file = "ablation_results.csv"

# initialize CSV if not exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=["Experiment","Epochs","Training Time (s)","Precision","Recall","mAP@0.5","mAP@0.5:0.95"]).to_csv(results_file,index=False)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [8]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(data=data_yaml, epochs=16, name="exp_epochs16", **base_params)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_epochs16",
    "Epochs": 16,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))



Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=16, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_epochs16, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 996.05it/s] 

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 66.7±21.2 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 502.34it/s]

...

                   all        230       1440      0.987      0.986      0.987      0.667
     DC Voltage Source        103        144      0.993      0.978      0.981      0.734
              Resistor        183        605      0.984      0.992      0.989      0.624
        Current Source        113        150       0.97      0.993      0.984      0.724
              Inductor        117        177      0.994      0.981      0.993      0.617
             Capacitor        115        184      0.989      0.989      0.986      0.598
     AC Voltage Source         65        180      0.993      0.983      0.991      0.706
Speed: 0.1ms preprocess, 1.8ms inference, 0.0ms loss, 5.9ms postprocess per image
Results saved to runs/detect/exp_epochs16
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 105.7±50.6 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 959.76it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.13it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.985      0.985      0.987      0.668
     DC Voltage Source        103        144      0.988      0.972      0.981      0.739
              Resistor        183        605      0.984      0.992      0.989      0.623
        Current Source        113        150      0.969      0.993      0.985      0.725
              Inductor        117        177      0.989      0.978      0.993      0.619
             Capacitor        115        184      0.989      0.989      0.986      0.599
     AC Voltage Source         65        180      0.993      0.983       0.99      0.703
Speed: 2.1ms preprocess, 4.2ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/exp_epochs162
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [9]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(data=data_yaml, epochs=32, name="exp_epochs32", **base_params)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_epochs32",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_epochs32, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 1000.79it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 32.0±29.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 562.48it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to runs/detect/exp_epochs32
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 104.6±57.6 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 986.70it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.14it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.2ms preprocess, 4.6ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/exp_epochs322
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [10]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(data=data_yaml, epochs=64, name="exp_epochs64", **base_params)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_epochs64",
    "Epochs": 64,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=64, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_epochs64, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 978.73it/s] 

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 51.6±27.8 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 586.50it/s]

...

                   all        230       1440      0.986      0.987      0.985      0.689
     DC Voltage Source        103        144      0.986      0.975      0.981      0.759
              Resistor        183        605      0.983      0.988      0.984      0.646
        Current Source        113        150      0.972      0.993      0.975      0.725
              Inductor        117        177      0.988      0.989       0.99      0.652
             Capacitor        115        184      0.992      0.989      0.987      0.621
     AC Voltage Source         65        180      0.994      0.987      0.992      0.729
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to runs/detect/exp_epochs64
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 55.6±35.8 MB/s, size: 58.5 K

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 945.96it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.25it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.986      0.987      0.985      0.689
     DC Voltage Source        103        144      0.986      0.975      0.981       0.76
              Resistor        183        605      0.983      0.988      0.984      0.646
        Current Source        113        150      0.972      0.993      0.975      0.727
              Inductor        117        177      0.988      0.989      0.991      0.653
             Capacitor        115        184      0.992      0.989      0.987       0.62
     AC Voltage Source         65        180      0.994      0.987      0.992      0.729
Speed: 2.1ms preprocess, 2.8ms inference, 0.0ms loss, 3.6ms postprocess per image
Results saved to runs/detect/exp_epochs642
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [11]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_wd0",
    **{**base_params, "weight_decay": 0}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_wd0",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_wd0, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100,

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 791.90it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 52.3±29.8 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 517.93it/s]

...

                   all        230       1440      0.986      0.989      0.986      0.679
     DC Voltage Source        103        144      0.984      0.979      0.978      0.751
              Resistor        183        605      0.983      0.992      0.986      0.634
        Current Source        113        150      0.974      0.997      0.984      0.736
              Inductor        117        177      0.989      0.988      0.989      0.632
             Capacitor        115        184      0.994      0.989      0.989      0.616
     AC Voltage Source         65        180      0.994      0.987      0.991      0.703
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.7ms postprocess per image
Results saved to runs/detect/exp_wd0
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 76.6±68.6 MB/s, size: 58.5 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 583.44it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.13it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.986      0.989      0.986      0.678
     DC Voltage Source        103        144      0.984      0.979      0.978       0.75
              Resistor        183        605      0.983      0.992      0.986      0.636
        Current Source        113        150      0.974      0.997      0.984      0.736
              Inductor        117        177      0.988      0.989      0.989      0.627
             Capacitor        115        184      0.994      0.989      0.989      0.615
     AC Voltage Source         65        180      0.994      0.987      0.991      0.706
Speed: 2.2ms preprocess, 4.9ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to runs/detect/exp_wd02
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+=======

In [12]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_wd0001",
    **{**base_params, "weight_decay": 0.0001}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_wd0001",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_wd0001, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 750.81it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 31.0±21.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 488.69it/s]

...

                   all        230       1440      0.986      0.986      0.987       0.68
     DC Voltage Source        103        144      0.984      0.972      0.981      0.745
              Resistor        183        605      0.982      0.985      0.983       0.64
        Current Source        113        150      0.973          1       0.98      0.726
              Inductor        117        177      0.988      0.989      0.993      0.643
             Capacitor        115        184      0.992      0.989      0.992      0.618
     AC Voltage Source         65        180      0.994      0.981      0.991      0.706
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 6.9ms postprocess per image
Results saved to runs/detect/exp_wd0001
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 82.5±42.6 MB/s, size: 58.5 KB)

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 895.89it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.12it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.986      0.986      0.987      0.678
     DC Voltage Source        103        144      0.984      0.972      0.981      0.745
              Resistor        183        605      0.982      0.985      0.983      0.641
        Current Source        113        150      0.973          1       0.98      0.727
              Inductor        117        177      0.988      0.989      0.993      0.639
             Capacitor        115        184      0.993      0.989      0.992      0.613
     AC Voltage Source         65        180      0.994      0.981      0.991      0.705
Speed: 2.4ms preprocess, 3.8ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/exp_wd00012
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+====

In [13]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_wd0005",
    **{**base_params, "weight_decay": 0.0005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_wd0005",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_wd0005, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 948.88it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 40.3±28.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 596.32it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to runs/detect/exp_wd0005
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 82.0±31.2 MB/s, size: 58.5 KB)

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 994.06it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.25it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.1ms preprocess, 3.6ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/exp_wd00052
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+====

In [14]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_wd001",
    **{**base_params, "weight_decay": 0.001}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_wd001",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_wd001, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=10

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 932.73it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 51.3±27.8 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 545.58it/s]

...

                   all        230       1440      0.987      0.987      0.988      0.682
     DC Voltage Source        103        144      0.992      0.972      0.985       0.76
              Resistor        183        605      0.985      0.985      0.986      0.638
        Current Source        113        150      0.974          1      0.984      0.733
              Inductor        117        177      0.983      0.989      0.988      0.642
             Capacitor        115        184      0.994      0.989      0.992      0.619
     AC Voltage Source         65        180      0.994      0.986      0.994      0.701
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/exp_wd001
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 87.7±60.7 MB/s, size: 58.5 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 885.03it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.12it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987      0.987      0.988      0.681
     DC Voltage Source        103        144      0.992      0.972      0.985      0.758
              Resistor        183        605      0.983      0.984      0.985       0.64
        Current Source        113        150      0.974          1      0.984      0.733
              Inductor        117        177      0.983      0.989      0.988      0.637
             Capacitor        115        184      0.994      0.989      0.992      0.617
     AC Voltage Source         65        180      0.994      0.986      0.994        0.7
Speed: 1.9ms preprocess, 4.9ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to runs/detect/exp_wd0012
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+=====

In [15]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_wd0005_high",
    **{**base_params, "weight_decay": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_wd0005_high",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_wd0005_high, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patie

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 1044.76it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 61.1±19.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 557.21it/s]


...

                   all        230       1440      0.988      0.987      0.988      0.678
     DC Voltage Source        103        144      0.984      0.972       0.98      0.756
              Resistor        183        605      0.985      0.993      0.988      0.639
        Current Source        113        150      0.978          1      0.992      0.738
              Inductor        117        177      0.993      0.989      0.991      0.632
             Capacitor        115        184      0.994      0.989      0.988       0.61
     AC Voltage Source         65        180      0.994      0.978      0.989      0.695
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to runs/detect/exp_wd0005_high
Ultralytics 8.3.182 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 79.4±51.4 MB/s, size: 58.

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 924.21it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.15it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.988      0.987      0.988      0.678
     DC Voltage Source        103        144      0.984      0.972       0.98      0.758
              Resistor        183        605      0.985      0.993      0.987      0.639
        Current Source        113        150      0.978          1      0.992      0.739
              Inductor        117        177      0.993      0.989      0.991      0.629
             Capacitor        115        184      0.994      0.989      0.988      0.607
     AC Voltage Source         65        180      0.994      0.978      0.989      0.696
Speed: 1.5ms preprocess, 4.8ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to runs/detect/exp_wd0005_high2
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+=

In [4]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_bs16",
    **{**base_params, "batch": 16, "weight_decay": 0.0005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_bs16",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_bs16, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3.7±1.6 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:06<00:00, 149.44it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5.4±1.5 MB/s, size: 35.4 KB)


...

                   all        230       1440      0.985      0.989      0.988      0.685
     DC Voltage Source        103        144      0.979      0.976      0.981      0.754
              Resistor        183        605      0.983      0.992      0.988      0.643
        Current Source        113        150      0.975          1      0.992      0.739
              Inductor        117        177      0.988      0.989      0.992       0.64
             Capacitor        115        184      0.989      0.989      0.985      0.617
     AC Voltage Source         65        180      0.994      0.989      0.989      0.717
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.4ms postprocess per image
Results saved to runs/detect/exp_bs16
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 107.8±76.1 MB/s, size: 58.5 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 888.69it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:03<00:00,  4.64it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.985      0.989      0.988      0.685
     DC Voltage Source        103        144      0.979      0.976      0.981      0.756
              Resistor        183        605      0.983      0.992      0.988      0.643
        Current Source        113        150      0.975          1      0.992      0.738
              Inductor        117        177      0.988      0.989      0.992      0.641
             Capacitor        115        184      0.989      0.989      0.985      0.617
     AC Voltage Source         65        180      0.994      0.989      0.989      0.715
Speed: 1.0ms preprocess, 4.8ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/exp_bs162
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+======

In [5]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_bs32",
    **{**base_params, "batch": 32, "weight_decay": 0.0005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_bs32",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_bs32, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 951.31it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 49.9±10.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 454.38it/s]


...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.7ms postprocess per image
Results saved to runs/detect/exp_bs32
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 89.0±50.6 MB/s, size: 58.5 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 932.45it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.23it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 1.9ms preprocess, 4.2ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to runs/detect/exp_bs322
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+======

In [6]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_bs64",
    **{**base_params, "batch": 64, "weight_decay": 0.0005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_bs64",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_bs64, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 998.99it/s] 

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 49.5±27.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 498.14it/s]

...

                   all        230       1440      0.983      0.988      0.988      0.683
     DC Voltage Source        103        144      0.993      0.971      0.984       0.75
              Resistor        183        605       0.98       0.99      0.986      0.629
        Current Source        113        150      0.966          1      0.985      0.737
              Inductor        117        177      0.985      0.989      0.993      0.641
             Capacitor        115        184      0.987      0.989      0.989       0.62
     AC Voltage Source         65        180      0.989       0.99      0.992      0.718
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to runs/detect/exp_bs64
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 74.4±46.7 MB/s, size: 58.5 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 891.96it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.11s/it]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.984      0.988      0.988      0.683
     DC Voltage Source        103        144      0.993      0.971      0.984       0.75
              Resistor        183        605      0.982      0.992      0.988       0.63
        Current Source        113        150      0.966          1      0.984      0.736
              Inductor        117        177      0.985      0.989      0.993      0.642
             Capacitor        115        184      0.987      0.989      0.989      0.621
     AC Voltage Source         65        180      0.989       0.99      0.993      0.721
Speed: 3.8ms preprocess, 1.8ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/exp_bs642
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+======

In [7]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_dropout0",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_dropout0",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_dropout0, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 917.96it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 40.5±23.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 508.83it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 6.7ms postprocess per image
Results saved to runs/detect/exp_dropout0
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 99.7±71.0 MB/s, size: 58.5 K

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 967.98it/s] 

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.24it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.0ms preprocess, 3.5ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/exp_dropout02
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [8]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_dropout01",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.1}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_dropout01",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_dropout01, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 1010.89it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 54.9±22.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 534.79it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.


...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/exp_dropout01
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 91.4±65.2 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 795.16it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.21it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.0ms preprocess, 4.0ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/exp_dropout012
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+=========

In [9]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_dropout02",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.2}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_dropout02",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_dropout022, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patien

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 995.21it/s] 

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 37.6±30.6 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 589.99it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to runs/detect/exp_dropout022
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 18.5±12.5 MB/s, size: 58.5

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:01<00:00, 122.80it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.02it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.0ms preprocess, 4.3ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/exp_dropout0222
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+========

In [10]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_dropout03",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.3}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_dropout03",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_dropout03, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:05<00:00, 155.87it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 26.0±25.2 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:01<00:00, 125.52it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to runs/detect/exp_dropout03
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 73.7±62.5 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 232.28it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.10it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.1ms preprocess, 4.1ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to runs/detect/exp_dropout032
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+=========

In [11]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_dropout05",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.5}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_dropout05",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)

print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.5, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_dropout05, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:04<00:00, 200.11it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4.1±2.7 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:01<00:00, 175.38it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/exp_dropout05
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 22.7±15.5 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:01<00:00, 166.81it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.04it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 2.1ms preprocess, 5.1ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to runs/detect/exp_dropout052
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+=========

In [12]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_img320",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 320}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_img320",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_img320, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 850.64it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 25.9±25.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 477.04it/s]

...

                   all        230       1440      0.983      0.987      0.987       0.67
     DC Voltage Source        103        144      0.992      0.979      0.976      0.743
              Resistor        183        605       0.98      0.992      0.989      0.616
        Current Source        113        150      0.954          1      0.979      0.728
              Inductor        117        177      0.994      0.989      0.993      0.634
             Capacitor        115        184      0.982      0.989      0.991      0.593
     AC Voltage Source         65        180      0.994      0.971      0.993      0.703
Speed: 0.0ms preprocess, 0.5ms inference, 0.0ms loss, 4.2ms postprocess per image
Results saved to runs/detect/exp_img320
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 11.4±12.0 MB/s, size: 58.5 KB)

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 395.88it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<00:00,  2.86it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.982      0.986      0.987      0.668
     DC Voltage Source        103        144      0.991      0.979      0.976      0.744
              Resistor        183        605      0.979      0.992      0.989      0.615
        Current Source        113        150      0.953          1      0.979      0.726
              Inductor        117        177      0.994      0.989      0.993      0.632
             Capacitor        115        184      0.982      0.989      0.991      0.588
     AC Voltage Source         65        180      0.994      0.968      0.993      0.705
Speed: 0.8ms preprocess, 5.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/detect/exp_img3202
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+==========+=

In [13]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_img512",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 512}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_img320",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_img512, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 563.26it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 21.9±19.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 363.48it/s]


...

                   all        230       1440       0.99      0.986      0.986      0.671
     DC Voltage Source        103        144          1      0.952       0.98      0.734
              Resistor        183        605      0.987      0.993      0.986      0.626
        Current Source        113        150      0.977          1      0.984      0.738
              Inductor        117        177      0.994      0.988      0.989       0.62
             Capacitor        115        184      0.993      0.989      0.984      0.613
     AC Voltage Source         65        180      0.992      0.994      0.994      0.695
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs/detect/exp_img512
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 31.7±26.0 MB/s, size: 58.5 KB)

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:01<00:00, 134.72it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.28it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.991      0.986      0.986       0.67
     DC Voltage Source        103        144          1      0.953       0.98      0.733
              Resistor        183        605      0.987      0.993      0.986      0.626
        Current Source        113        150      0.977          1      0.984      0.737
              Inductor        117        177      0.994      0.988      0.989       0.62
             Capacitor        115        184      0.993      0.989      0.984      0.609
     AC Voltage Source         65        180      0.992      0.994      0.992      0.693
Speed: 1.4ms preprocess, 4.0ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/exp_img5122
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+==========+=

In [ ]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_img640",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_img320",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_img640, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 844.59it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 42.2±20.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 443.56it/s]

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      28/32      4.61G       1.11     0.5342      1.127        136        640: 100%|██████████| 29/29 [00:09<00:00,  3.11it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  3.20it/s]

                   all        230       1440      0.987      0.986      0.986      0.669



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      29/32      4.62G      1.092     0.5207      1.121        145        640: 100%|██████████| 29/29 [00:09<00:00,  3.18it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  25%|██▌       | 1/4 [00:00<00:01,  2.74it/s]

In [5]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_img640",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_img320",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_img640, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 16.1±9.8 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:02<00:00, 367.87it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.


...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to runs/detect/exp_img640
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 108.6±103.9 MB/s, size: 58.5 K

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 898.03it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.25it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 1.7ms preprocess, 4.8ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to runs/detect/exp_img6402
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+====

In [6]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_opt_sgd",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "SGD", "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_opt_sgd",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_opt_sgd, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 880.78it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 58.5±23.8 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 431.24it/s]

...

                   all        230       1440      0.981      0.989      0.988      0.682
     DC Voltage Source        103        144      0.979      0.978      0.974      0.745
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991      0.736
              Inductor        117        177      0.979      0.989      0.991      0.638
             Capacitor        115        184      0.987      0.989      0.991      0.618
     AC Voltage Source         65        180      0.988      0.989      0.994       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/exp_opt_sgd
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 116.1±103.0 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 901.48it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.25it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.989      0.988      0.683
     DC Voltage Source        103        144      0.979      0.978      0.975      0.746
              Resistor        183        605       0.98      0.991      0.985      0.634
        Current Source        113        150      0.973          1      0.991       0.74
              Inductor        117        177      0.979      0.989      0.991       0.64
             Capacitor        115        184      0.987      0.989      0.991      0.619
     AC Voltage Source         65        180      0.988      0.989      0.994      0.716
Speed: 1.9ms preprocess, 4.9ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs/detect/exp_opt_sgd2
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+===

In [7]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_opt_Adam",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adam", "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_opt_Adam",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_opt_Adam, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patienc

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 810.67it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 50.3±26.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 497.43it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.


...

                   all        230       1440      0.984      0.985      0.987      0.687
     DC Voltage Source        103        144      0.985      0.979      0.984      0.761
              Resistor        183        605      0.983      0.983      0.985      0.645
        Current Source        113        150      0.971      0.993      0.989      0.743
              Inductor        117        177      0.976      0.989      0.984      0.633
             Capacitor        115        184      0.995      0.989       0.99      0.613
     AC Voltage Source         65        180      0.994      0.976      0.992      0.724
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/exp_opt_Adam
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 105.1±90.4 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 909.83it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.01it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.984      0.985      0.987      0.686
     DC Voltage Source        103        144      0.985      0.979      0.984      0.761
              Resistor        183        605      0.983      0.985      0.985      0.645
        Current Source        113        150      0.972      0.993      0.989      0.738
              Inductor        117        177      0.977      0.989      0.984      0.636
             Capacitor        115        184      0.995      0.988       0.99      0.616
     AC Voltage Source         65        180      0.994      0.976      0.992       0.72
Speed: 2.0ms preprocess, 5.1ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/exp_opt_Adam2
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [9]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_opt_AdamW",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "AdamW", "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_opt_AdamW",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_opt_AdamW, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patie

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 891.18it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 45.8±31.2 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 489.13it/s]

...

                   all        230       1440      0.985      0.983      0.987      0.685
     DC Voltage Source        103        144      0.997      0.965      0.983      0.761
              Resistor        183        605      0.984      0.986      0.986      0.645
        Current Source        113        150      0.966      0.993      0.985      0.733
              Inductor        117        177      0.989      0.989       0.99      0.642
             Capacitor        115        184      0.984      0.989      0.988       0.61
     AC Voltage Source         65        180      0.989      0.978      0.991      0.719
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to runs/detect/exp_opt_AdamW
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 105.9±96.6 MB/s, size: 58.5

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 928.71it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.15it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.985      0.983      0.987      0.685
     DC Voltage Source        103        144      0.997      0.965      0.983      0.762
              Resistor        183        605      0.984      0.987      0.986      0.645
        Current Source        113        150      0.966      0.993      0.985      0.735
              Inductor        117        177      0.989      0.989       0.99      0.641
             Capacitor        115        184      0.984      0.988      0.988      0.609
     AC Voltage Source         65        180      0.989      0.978      0.991       0.72
Speed: 2.1ms preprocess, 3.5ms inference, 0.0ms loss, 3.6ms postprocess per image
Results saved to runs/detect/exp_opt_AdamW2
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+=========

In [10]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_opt_RMSProp",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "RMSProp", "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_opt_RMSProp",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_opt_RMSProp, nbs=64, nms=False, opset=None, optimize=False, optimizer=RMSProp, overlap_mask=True, p

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 948.27it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 44.5±22.2 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 503.01it/s]

...

                   all        230       1440      0.612       0.67      0.646      0.257
     DC Voltage Source        103        144      0.513      0.778      0.692      0.344
              Resistor        183        605      0.799      0.896      0.851      0.288
        Current Source        113        150      0.503      0.649      0.571      0.277
              Inductor        117        177      0.481      0.644      0.461      0.125
             Capacitor        115        184      0.602      0.516      0.536      0.159
     AC Voltage Source         65        180      0.776      0.538      0.766      0.352
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to runs/detect/exp_opt_RMSProp
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 88.9±69.9 MB/s, size: 58.

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 889.29it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.99it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.609      0.667      0.644      0.256
     DC Voltage Source        103        144      0.517      0.771      0.692      0.344
              Resistor        183        605      0.802      0.901      0.857      0.286
        Current Source        113        150      0.501      0.642      0.569      0.275
              Inductor        117        177      0.478      0.644       0.46      0.124
             Capacitor        115        184       0.59      0.505      0.521      0.158
     AC Voltage Source         65        180      0.769      0.536      0.763      0.351
Speed: 2.1ms preprocess, 3.9ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/exp_opt_RMSProp2
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+=

In [11]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_opt_Adamax",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adamax", "imgsz": 640}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_opt_Adamax",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_opt_Adamax, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, pat

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 760.98it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 42.9±29.8 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 440.19it/s]

...

                   all        230       1440      0.988      0.986      0.988      0.691
     DC Voltage Source        103        144      0.991      0.979      0.982      0.773
              Resistor        183        605      0.984       0.99      0.988      0.643
        Current Source        113        150      0.979          1      0.987      0.741
              Inductor        117        177      0.994      0.977      0.992      0.645
             Capacitor        115        184      0.984      0.989      0.988      0.617
     AC Voltage Source         65        180      0.994       0.98      0.991      0.727
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to runs/detect/exp_opt_Adamax
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 106.4±99.0 MB/s, size: 58.

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 833.18it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.27it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.988      0.986      0.988       0.69
     DC Voltage Source        103        144      0.991      0.979      0.982      0.771
              Resistor        183        605      0.984       0.99      0.988      0.641
        Current Source        113        150      0.979          1      0.987      0.741
              Inductor        117        177      0.994      0.977      0.992      0.641
             Capacitor        115        184      0.984      0.989      0.988      0.618
     AC Voltage Source         65        180      0.994       0.98      0.991      0.729
Speed: 2.0ms preprocess, 4.1ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/exp_opt_Adamax2
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+==

In [12]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_lr0001",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adamax", "imgsz": 640, "lr0": 0.001}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_lr0001",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_lr0001, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patien

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 789.83it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 47.2±26.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 504.81it/s]

...

                   all        230       1440      0.986      0.987      0.987      0.679
     DC Voltage Source        103        144      0.982      0.979      0.974       0.75
              Resistor        183        605      0.985       0.99       0.99      0.635
        Current Source        113        150      0.974      0.993      0.991      0.742
              Inductor        117        177      0.987      0.989      0.993      0.629
             Capacitor        115        184      0.994      0.989      0.984      0.608
     AC Voltage Source         65        180      0.994      0.979       0.99      0.709
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/exp_lr0001
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 107.1±115.9 MB/s, size: 58.5 K

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 791.76it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.27it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987      0.986      0.987      0.679
     DC Voltage Source        103        144      0.985      0.979      0.974       0.75
              Resistor        183        605      0.985      0.988       0.99      0.636
        Current Source        113        150      0.977      0.993      0.991      0.741
              Inductor        117        177      0.989      0.989      0.993       0.63
             Capacitor        115        184      0.995      0.987      0.984      0.609
     AC Voltage Source         65        180      0.994      0.979       0.99      0.709
Speed: 2.0ms preprocess, 4.0ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to runs/detect/exp_lr00012
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+======

In [14]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_lr0005",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adamax", "imgsz": 640, "lr0": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_lr0005",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_lr00052, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patie

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 970.46it/s] 

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 41.8±21.1 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 569.57it/s]


...

                   all        230       1440      0.984      0.988      0.988      0.691
     DC Voltage Source        103        144      0.979      0.978       0.98      0.761
              Resistor        183        605      0.982      0.993      0.987      0.647
        Current Source        113        150      0.974          1      0.991      0.742
              Inductor        117        177      0.983      0.987      0.992      0.657
             Capacitor        115        184      0.992      0.989      0.987      0.618
     AC Voltage Source         65        180      0.994      0.981       0.99       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.9ms postprocess per image
Results saved to runs/detect/exp_lr00052
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 115.5±97.9 MB/s, size: 58.5 K

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 925.23it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.84it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.984      0.988      0.988      0.691
     DC Voltage Source        103        144      0.979      0.978       0.98      0.762
              Resistor        183        605      0.982      0.993      0.987      0.647
        Current Source        113        150      0.974          1      0.991       0.74
              Inductor        117        177      0.983      0.987      0.992      0.658
             Capacitor        115        184      0.992      0.989      0.987       0.62
     AC Voltage Source         65        180      0.994      0.981       0.99      0.719
Speed: 2.1ms preprocess, 3.8ms inference, 0.0ms loss, 4.3ms postprocess per image
Results saved to runs/detect/exp_lr000522
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+=====

In [15]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_lr001",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adamax", "imgsz": 640, "lr0": 0.01}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_lr001",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_lr001, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 896.09it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 45.9±30.4 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 575.05it/s]

...

                   all        230       1440      0.988      0.986      0.988      0.691
     DC Voltage Source        103        144      0.991      0.979      0.982      0.773
              Resistor        183        605      0.984       0.99      0.988      0.643
        Current Source        113        150      0.979          1      0.987      0.741
              Inductor        117        177      0.994      0.977      0.992      0.645
             Capacitor        115        184      0.984      0.989      0.988      0.617
     AC Voltage Source         65        180      0.994       0.98      0.991      0.727
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/exp_lr001
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 129.3±110.4 MB/s, size: 58.5 KB

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 952.44it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.20it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.988      0.986      0.988       0.69
     DC Voltage Source        103        144      0.991      0.979      0.982      0.771
              Resistor        183        605      0.984       0.99      0.988      0.641
        Current Source        113        150      0.979          1      0.987      0.741
              Inductor        117        177      0.994      0.977      0.992      0.641
             Capacitor        115        184      0.984      0.989      0.988      0.618
     AC Voltage Source         65        180      0.994       0.98      0.991      0.729
Speed: 2.0ms preprocess, 3.6ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/exp_lr0012
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+=======

In [16]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_lr005",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adamax", "imgsz": 640, "lr0": 0.05}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_lr005",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_lr005, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 897.74it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 27.4±23.4 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 536.73it/s]

...

                   all        230       1440      0.987      0.983      0.987      0.679
     DC Voltage Source        103        144          1      0.961      0.982      0.748
              Resistor        183        605      0.985       0.98      0.988      0.633
        Current Source        113        150      0.969          1       0.98      0.728
              Inductor        117        177      0.988      0.989      0.993       0.64
             Capacitor        115        184      0.988      0.989      0.987      0.611
     AC Voltage Source         65        180      0.994       0.98      0.994      0.714
Speed: 0.3ms preprocess, 1.5ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/exp_lr005
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 117.9±110.7 MB/s, size: 58.5 KB

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 857.15it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.09it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987      0.983      0.987       0.68
     DC Voltage Source        103        144          1      0.961      0.982      0.752
              Resistor        183        605      0.985      0.981      0.987      0.633
        Current Source        113        150      0.969          1       0.98      0.728
              Inductor        117        177      0.988      0.989      0.993      0.641
             Capacitor        115        184      0.988      0.989      0.987      0.612
     AC Voltage Source         65        180      0.994       0.98      0.994      0.714
Speed: 3.0ms preprocess, 5.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs/detect/exp_lr0052
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+=======

In [17]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_lr01",
    **{**base_params, "batch": 32, "weight_decay": 0.0005, "dropout": 0.0, "optimizer": "Adamax", "imgsz": 640, "lr0": 0.1}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_lr01",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_lr01, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patience=1

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 933.80it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 47.7±24.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 546.12it/s]

...

                   all        230       1440      0.957      0.972      0.983      0.662
     DC Voltage Source        103        144      0.945      0.962      0.977       0.74
              Resistor        183        605      0.976      0.989      0.988      0.632
        Current Source        113        150      0.908      0.984      0.971      0.702
              Inductor        117        177       0.95      0.955      0.981        0.6
             Capacitor        115        184      0.981      0.984      0.988      0.579
     AC Voltage Source         65        180      0.983      0.956       0.99      0.716
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 6.3ms postprocess per image
Results saved to runs/detect/exp_lr01
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 125.5±93.4 MB/s, size: 58.5 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 932.27it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.12it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.957      0.972      0.983      0.661
     DC Voltage Source        103        144      0.945      0.963      0.977      0.742
              Resistor        183        605      0.975       0.99      0.988       0.63
        Current Source        113        150      0.908      0.985      0.971      0.701
              Inductor        117        177      0.949      0.955      0.981      0.599
             Capacitor        115        184      0.981      0.984      0.988       0.58
     AC Voltage Source         65        180      0.983      0.957       0.99      0.713
Speed: 2.2ms preprocess, 4.4ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/exp_lr012
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+========

In [18]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_freeze0",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005, "freeze": 0}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_freeze0",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_freeze0, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patience

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 928.21it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 40.7±31.1 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 489.00it/s]

...

                   all        230       1440      0.984      0.988      0.988      0.691
     DC Voltage Source        103        144      0.979      0.978       0.98      0.761
              Resistor        183        605      0.982      0.993      0.987      0.647
        Current Source        113        150      0.974          1      0.991      0.742
              Inductor        117        177      0.983      0.987      0.992      0.657
             Capacitor        115        184      0.992      0.989      0.987      0.618
     AC Voltage Source         65        180      0.994      0.981       0.99       0.72
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 6.3ms postprocess per image
Results saved to runs/detect/exp_freeze0
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 123.1±111.2 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 818.99it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.11it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.984      0.988      0.988      0.691
     DC Voltage Source        103        144      0.979      0.978       0.98      0.762
              Resistor        183        605      0.982      0.993      0.987      0.647
        Current Source        113        150      0.974          1      0.991       0.74
              Inductor        117        177      0.983      0.987      0.992      0.658
             Capacitor        115        184      0.992      0.989      0.987       0.62
     AC Voltage Source         65        180      0.994      0.981       0.99      0.719
Speed: 2.0ms preprocess, 4.9ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to runs/detect/exp_freeze02
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+=====

In [19]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_freeze10",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005, "freeze": 10}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_freeze10",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_freeze10, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patien

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 890.98it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 33.2±33.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 495.57it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.


...

                   all        230       1440      0.981      0.985      0.988      0.681
     DC Voltage Source        103        144      0.967      0.979      0.983      0.757
              Resistor        183        605      0.984      0.991      0.988      0.636
        Current Source        113        150       0.98       0.99      0.988      0.728
              Inductor        117        177      0.978      0.983      0.985      0.634
             Capacitor        115        184      0.989      0.987       0.99      0.624
     AC Voltage Source         65        180      0.989       0.98      0.991      0.709
Speed: 0.2ms preprocess, 1.9ms inference, 0.0ms loss, 6.1ms postprocess per image
Results saved to runs/detect/exp_freeze10
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 125.8±83.7 MB/s, size: 58.5 

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 913.07it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.18it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.981      0.985      0.988      0.681
     DC Voltage Source        103        144      0.967      0.979      0.983      0.758
              Resistor        183        605      0.984      0.991      0.988      0.636
        Current Source        113        150       0.98      0.992      0.988      0.726
              Inductor        117        177      0.977      0.983      0.985      0.635
             Capacitor        115        184      0.989      0.987       0.99      0.624
     AC Voltage Source         65        180      0.989       0.98      0.991      0.707
Speed: 2.0ms preprocess, 3.7ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/exp_freeze102
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+====

In [20]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_freeze15",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005, "freeze": 15}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_freeze15",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=15, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_freeze15, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patien

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:00<00:00, 924.41it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 38.7±29.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 474.96it/s]

...

                   all        230       1440      0.979      0.978      0.985      0.671
     DC Voltage Source        103        144       0.97      0.972      0.979      0.745
              Resistor        183        605      0.982      0.984      0.988      0.644
        Current Source        113        150      0.975       0.98      0.987      0.722
              Inductor        117        177      0.971      0.989      0.989      0.613
             Capacitor        115        184      0.989      0.959      0.978      0.597
     AC Voltage Source         65        180      0.986      0.983       0.99      0.703
Speed: 0.1ms preprocess, 1.9ms inference, 0.0ms loss, 4.3ms postprocess per image
Results saved to runs/detect/exp_freeze15
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 130.0±109.7 MB/s, size: 58.5

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 928.38it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.12it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.979      0.978      0.985      0.672
     DC Voltage Source        103        144      0.969      0.972      0.979      0.746
              Resistor        183        605      0.982      0.983      0.988      0.645
        Current Source        113        150      0.975       0.98      0.987      0.726
              Inductor        117        177      0.973      0.989      0.989      0.614
             Capacitor        115        184      0.989      0.961      0.978      0.596
     AC Voltage Source         65        180      0.986      0.983       0.99      0.702
Speed: 2.0ms preprocess, 4.2ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/exp_freeze152
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+====

In [21]:
model = YOLO("yolov8n.pt")

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_freeze22",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005, "freeze": 22}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_freeze22",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_freeze22, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patien

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 830.92it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 51.1±25.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 561.28it/s]

...

                   all        230       1440       0.93      0.952      0.972      0.626
     DC Voltage Source        103        144      0.927      0.958       0.97      0.708
              Resistor        183        605      0.977      0.974      0.986      0.607
        Current Source        113        150      0.836      0.951      0.966      0.669
              Inductor        117        177      0.915      0.932      0.958      0.536
             Capacitor        115        184      0.983      0.967      0.982      0.555
     AC Voltage Source         65        180      0.944       0.93      0.973      0.678
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/exp_freeze22
Ultralytics 8.3.183 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 134.8±109.8 MB/s, size: 58.5

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 921.78it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.14it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.931      0.949      0.972      0.625
     DC Voltage Source        103        144      0.929      0.958      0.971      0.708
              Resistor        183        605      0.978      0.974      0.987      0.607
        Current Source        113        150      0.835       0.94      0.965      0.669
              Inductor        117        177      0.921      0.921      0.956      0.536
             Capacitor        115        184      0.983      0.966      0.982      0.555
     AC Voltage Source         65        180      0.943      0.933      0.973      0.677
Speed: 1.9ms preprocess, 3.4ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/exp_freeze222
+----+-----------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment      |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+=================+====

In [3]:
import os

# Path to base YOLOv8 YAML from ultralytics package
base_yaml_path = "/kaggle/working/yolov8n.yaml"

# First, save the default YOLOv8n.yaml from ultralytics
from ultralytics.nn.tasks import yaml_model_load
import yaml

# Load YOLOv8n.yaml structure
cfg = yaml_model_load("yolov8n.yaml")

# Function to create variants with different activation functions
def create_yaml_with_activation(act_name):
    cfg_mod = cfg.copy()
    cfg_mod['activation'] = act_name  # swap activation
    out_path = f"/kaggle/working/yolov8n_{act_name.lower()}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg_mod, f)
    print(f"✅ Created {out_path}")
    return out_path

# Create ReLU, LeakyReLU, Mish YAMLs
relu_yaml = create_yaml_with_activation("ReLU")
leaky_yaml = create_yaml_with_activation("LeakyReLU")
mish_yaml = create_yaml_with_activation("Mish")


✅ Created /kaggle/working/yolov8n_relu.yaml
✅ Created /kaggle/working/yolov8n_leakyrelu.yaml
✅ Created /kaggle/working/yolov8n_mish.yaml


In [4]:
model = YOLO(relu_yaml)

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_act_relu",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_act_relu",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


NameError: name 'ReLU' is not defined

In [7]:
import yaml
from ultralytics.nn.tasks import yaml_model_load

# Load base config
cfg = yaml_model_load("yolov8n.yaml")

def create_yaml_with_activation(act_expr):
    cfg_mod = cfg.copy()
    cfg_mod['activation'] = act_expr
    out_path = f"/kaggle/working/yolov8n_{act_expr.replace('nn.','').replace('()','').lower()}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg_mod, f)
    print(f"✅ Created {out_path}")
    return out_path

relu_yaml = create_yaml_with_activation("nn.ReLU()")
leaky_yaml = create_yaml_with_activation("nn.LeakyReLU(0.1)")
mish_yaml = create_yaml_with_activation("nn.Mish()")
silu_yaml = create_yaml_with_activation("nn.SiLU()")  # baseline


✅ Created /kaggle/working/yolov8n_relu.yaml
✅ Created /kaggle/working/yolov8n_leakyrelu(0.1).yaml
✅ Created /kaggle/working/yolov8n_mish.yaml
✅ Created /kaggle/working/yolov8n_silu.yaml


In [8]:
model = YOLO(relu_yaml)

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_act_relu",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_act_relu",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_relu.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_act_relu, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, 

Overriding model.yaml nc=80 with nc=6
activation: nn.ReLU()

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1 

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 17.4±9.7 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:03<00:00, 303.41it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 12.0±5.6 MB/s, size: 35.4 KB)


...

                   all        230       1440      0.976      0.971      0.984       0.66
     DC Voltage Source        103        144      0.989      0.938      0.978      0.736
              Resistor        183        605      0.982       0.98      0.985      0.611
        Current Source        113        150       0.94      0.987      0.984      0.723
              Inductor        117        177      0.977      0.971       0.98      0.609
             Capacitor        115        184      0.994      0.973      0.986      0.583
     AC Voltage Source         65        180      0.972      0.978      0.993      0.699
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 6.5ms postprocess per image
Results saved to runs/detect/exp_act_relu
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8n_relu summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 96.8±72.2 MB/s, size:

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 823.07it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.94it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.976      0.972      0.985       0.66
     DC Voltage Source        103        144      0.989      0.938      0.978      0.736
              Resistor        183        605      0.985      0.983      0.988      0.614
        Current Source        113        150       0.94      0.987      0.984      0.723
              Inductor        117        177      0.976      0.972       0.98      0.607
             Capacitor        115        184      0.994      0.973      0.986      0.586
     AC Voltage Source         65        180      0.971      0.978      0.993      0.697
Speed: 2.0ms preprocess, 4.0ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/exp_act_relu2
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [9]:
model = YOLO(mish_yaml)



start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_act_mish",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_act_mish",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))

Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_mish.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_act_mish, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, 

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 874.19it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 55.7±15.1 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 465.85it/s]

...

                   all        230       1440       0.98      0.971      0.985      0.666
     DC Voltage Source        103        144          1      0.938       0.98      0.749
              Resistor        183        605      0.977      0.984      0.983      0.619
        Current Source        113        150      0.944       0.98      0.987      0.717
              Inductor        117        177      0.967      0.966      0.987      0.601
             Capacitor        115        184      0.995      0.987      0.983      0.597
     AC Voltage Source         65        180      0.994      0.971      0.993      0.712
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to runs/detect/exp_act_mish
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8n_mish summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 100.7±59.9 MB/s, size

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 824.39it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.00it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440       0.98      0.971      0.985      0.666
     DC Voltage Source        103        144          1      0.938       0.98      0.751
              Resistor        183        605      0.977      0.985      0.983      0.619
        Current Source        113        150      0.944       0.98      0.987      0.717
              Inductor        117        177      0.967      0.966      0.987      0.603
             Capacitor        115        184      0.995      0.987      0.983      0.598
     AC Voltage Source         65        180      0.994      0.971      0.993      0.711
Speed: 2.1ms preprocess, 4.9ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/exp_act_mish2
+----+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+==

In [10]:
model = YOLO(leaky_yaml)

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_act_leaky",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_act_leaky",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_leakyrelu(0.1).yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_act_leaky, nbs=64, nms=False, opset=None, optimize=False, optimiz

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 860.27it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 41.2±23.3 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 472.17it/s]

...

                   all        230       1440      0.971      0.971      0.984      0.664
     DC Voltage Source        103        144      0.971      0.972      0.978      0.739
              Resistor        183        605       0.98      0.983      0.987      0.619
        Current Source        113        150       0.95      0.967      0.978      0.721
              Inductor        117        177      0.956      0.976      0.983      0.598
             Capacitor        115        184      0.994      0.963      0.986      0.603
     AC Voltage Source         65        180      0.978      0.968       0.99      0.707
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/exp_act_leaky
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8n_leakyrelu(0.1) summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 102.2±74.3

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 786.03it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.98it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.972      0.973      0.984      0.664
     DC Voltage Source        103        144       0.97      0.972      0.978      0.739
              Resistor        183        605      0.982      0.987      0.988       0.62
        Current Source        113        150       0.95      0.967      0.978      0.722
              Inductor        117        177      0.958      0.977      0.984      0.595
             Capacitor        115        184      0.994      0.963      0.986      0.606
     AC Voltage Source         65        180      0.978      0.972      0.989      0.704
Speed: 2.1ms preprocess, 4.0ms inference, 0.0ms loss, 3.9ms postprocess per image
Results saved to runs/detect/exp_act_leaky2
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+=========

In [11]:
model = YOLO(silu_yaml)

start = time.time()
model.train(
    data=data_yaml,
    epochs=32,
    name="exp_act_silu",
    **{**base_params, "batch": 32, "optimizer": "Adamax", "weight_decay": 0.0005, "dropout": 0.0, "imgsz": 640, "lr0": 0.005}
)
end = time.time()

val = model.val()
result = {
    "Experiment": "exp_act_silu",
    "Epochs": 32,
    "Training Time (s)": round(end - start, 2),
    "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
    "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
    "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
    "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
}

df = pd.read_csv(results_file)
df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
df.to_csv(results_file,index=False)
print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_silu.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_act_silu, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, 

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 885.80it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 45.6±25.9 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 517.86it/s]

...

                   all        230       1440      0.976      0.982      0.985      0.668
     DC Voltage Source        103        144      0.991      0.972      0.979      0.744
              Resistor        183        605      0.985       0.98      0.988      0.636
        Current Source        113        150      0.949      0.993      0.974      0.722
              Inductor        117        177      0.965      0.983      0.989      0.607
             Capacitor        115        184      0.986      0.989      0.988      0.595
     AC Voltage Source         65        180      0.983      0.976       0.99      0.702
Speed: 0.2ms preprocess, 2.0ms inference, 0.0ms loss, 4.2ms postprocess per image
Results saved to runs/detect/exp_act_silu
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8n_silu summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 84.7±52.6 MB/s, size:

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 774.46it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.97it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.976      0.981      0.985      0.666
     DC Voltage Source        103        144      0.991      0.972      0.979      0.741
              Resistor        183        605      0.985      0.979      0.988      0.634
        Current Source        113        150      0.949      0.989      0.974      0.721
              Inductor        117        177      0.965      0.983      0.989      0.606
             Capacitor        115        184      0.986      0.989      0.988      0.593
     AC Voltage Source         65        180      0.983      0.976       0.99      0.704
Speed: 2.2ms preprocess, 5.1ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/exp_act_silu2
+----+---------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment    |   Epochs |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+===============+==========

In [12]:
# Case 11: Compare YOLOv8 sizes (n, s, m, l)

models = {
    "exp_size_n": "yolov8n.yaml",
    "exp_size_s": "yolov8s.yaml",
    "exp_size_m": "yolov8m.yaml",
    "exp_size_l": "yolov8l.yaml"
}

for exp_name, cfg in models.items():
    model = YOLO(cfg)
    
    start = time.time()
    model.train(
        data=data_yaml,
        epochs=32,
        name=exp_name,
        **{**base_params, 
           "batch": 32, 
           "optimizer": "Adamax", 
           "weight_decay": 0.0005, 
           "dropout": 0.0, 
           "imgsz": 640, 
           "lr0": 0.005}
    )
    end = time.time()
    
    # Validate
    val = model.val()
    
    result = {
        "Experiment": exp_name,
        "Epochs": 32,
        "Training Time (s)": round(end - start, 2),
        "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
        "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
        "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
        "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
    }
    
    # Save results
    df = pd.read_csv(results_file)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
    df.to_csv(results_file, index=False)
    print(tabulate(df, headers="keys", tablefmt="grid"))


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp_size_n, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, pati

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 851.99it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 39.3±26.4 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 468.47it/s]

...

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 487.89it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.


Plotting labels to runs/detect/exp_size_l/labels.jpg... 
optimizer: Adamax(lr=0.005, momentum=0.937) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to runs/detect/exp_size_l
Starting training for 32 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/29 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 26.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 21.12 MiB is free. Process 3581 has 15.86 GiB memory in use. Of the allocated memory 15.46 GiB is allocated by PyTorch, and 83.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [15]:
!zip -r /kaggle/working/working_dir.zip /kaggle/working/


updating: kaggle/working/ (stored 0%)
updating: kaggle/working/runs/ (stored 0%)
updating: kaggle/working/runs/detect/ (stored 0%)
updating: kaggle/working/runs/detect/exp_act_mish/ (stored 0%)
updating: kaggle/working/runs/detect/exp_act_mish/BoxPR_curve.png (deflated 17%)
updating: kaggle/working/runs/detect/exp_act_mish/args.yaml (deflated 52%)
updating: kaggle/working/runs/detect/exp_act_mish/val_batch2_labels.jpg (deflated 21%)
updating: kaggle/working/runs/detect/exp_act_mish/train_batch640.jpg (deflated 23%)
updating: kaggle/working/runs/detect/exp_act_mish/BoxF1_curve.png (deflated 8%)
updating: kaggle/working/runs/detect/exp_act_mish/val_batch1_pred.jpg (deflated 14%)
updating: kaggle/working/runs/detect/exp_act_mish/labels_correlogram.jpg (deflated 38%)
updating: kaggle/working/runs/detect/exp_act_mish/labels.jpg (deflated 28%)
updating: kaggle/working/runs/detect/exp_act_mish/weights/ (stored 0%)
updating: kaggle/working/runs/detect/exp_act_mish/weights/last.pt (deflated 19%

In [16]:
from kaggle import files
files.download('/kaggle/working/working_dir.zip')


OSError: Could not find kaggle.json. Make sure it's located in /root/.config/kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/

In [17]:
import shutil

# (1) Zip your working folder
shutil.make_archive('/kaggle/working/working_dir', 'zip', '/kaggle/working')

# (2) Use IPython display to download
from IPython.display import FileLink
FileLink('/kaggle/working/working_dir.zip')


/kaggle/working/working_dir.zip